In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

train_path = "/kaggle/input/q1-stage-3-2026/PlantVillage/train"
test_path = "/kaggle/input/q1-stage-3-2026/PlantVillage/test"

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dataset =ImageFolder(train_path, train_transform)
test_dataset = ImageFolder(test_path, test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print(f"Training dataset size: {len(train_dataset)}, testing dataset size: {len(test_dataset)}")

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

class_mapping = {
    0: "Early Blight",
    1: "Late Blight",
    2: "Healthy"
}

random_indices = random.sample(range(len(train_dataset)), 4)



fig, axes = plt.subplots(1, 4, figsize=(10, 5))

for i, idx in enumerate(random_indices):
  image, label = train_dataset[idx]
  image = image.permute(1, 2, 0).numpy()
  ax = axes[i]
  ax.imshow(image)
  ax.set_title(f"Label:{class_mapping[label]}")
  ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
class CNNModel(nn.Module):
  def __init__(self, num_classes):
    super(CNNModel, self).__init__()

    self.feature_extractor = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1), #(batch, 16, 16, 16)
        nn.BatchNorm2d(16),
        nn.ReLU(inplace=True),

        nn.Conv2d(16, 32, kernel_size=3, padding=1), #(batch, 32, 16, 16)
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True),

        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), #(batch, 64, 8, 8)
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),

        nn.Conv2d(64, 128, kernel_size=3, padding=1), #(batch, 128, 8, 8)
        nn.BatchNorm2d(128),
        nn.ReLU(inplace=True),

        nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1), #(batch, 256, 4, 4)
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256*4*4, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU(inplace=True),
        nn.Linear(4096, num_classes)
    )


  def forward(self, x):
    features = self.feature_extractor(x)
    classification = self.classifier(features)
    return classification

In [ ]:
# Write your code here
# Write your code here
from tqdm import tqdm

def train_one_epoch(model, train_loader, optimizer, criterion, device):
  model.train()
  total_loss = 0
  correct= 0
  total = 0

  for images, labels in tqdm(train_loader):
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    probabilities = torch.softmax(outputs, dim=1)
    predictions = probabilities.argmax(dim=1)

    correct += (predictions==labels).sum().item()
    total_loss += loss.item()
    total += labels.size(0)

  avg_loss = total_loss/len(train_loader)
  acc = 100*correct/total

  return avg_loss, acc

In [ ]:
def validate(model, val_loader, criterion, device):
  model.eval()
  total_loss = 0
  correct = 0
  total = 0

  with torch.no_grad():
    for images, labels in tqdm(val_loader):
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)

      loss = criterion(outputs, labels)

      probabilities = torch.softmax(outputs, dim=1)
      predictions = probabilities.argmax(dim=1)

      total_loss += loss.item()
      correct += (predictions==labels).sum().item()
      total += labels.size(0)

  avg_loss = total_loss/len(val_loader)
  acc= 100*correct/total
  return avg_loss, acc


In [ ]:
# Write your code here
import torch.optim as optim

model = CNNModel(num_classes=3) #already defined in the previous code
LEARNING_RATE = .001
NUM_EPOCHS= 10
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):
  train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
  val_loss, val_acc = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)
  train_accuracies.append(train_acc)
  val_accuracies.append(val_acc)

  print(f"Epoch[{epoch+1}/{NUM_EPOCHS}]: Training Loss: {train_loss} | Validation Loss: {val_loss} | Training Accuracy: {train_acc} | Validation Accuracy: {val_acc}")

In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, NUM_EPOCHS+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, NUM_EPOCHS+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, NUM_EPOCHS+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, NUM_EPOCHS+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
class CNNResidual(nn.Module):
  def __init__(self, num_classes):
    super(CNNResidual, self).__init__()

    self.conv1 = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1), #(batch, 16, 16, 16)
        nn.BatchNorm2d(16),
        nn.ReLU(inplace=True)
    )

    self.conv2 = nn.Sequential(
        nn.Conv2d(16, 32, kernel_size=3, padding=1), #(batch, 32, 16, 16)
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True)
    )

    self.conv3 = nn.Sequential(
        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), #(batch, 64, 8, 8)
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True)
    )

    self.conv4 = nn.Sequential(
        nn.Conv2d(96, 128, kernel_size=3, padding=1), #(batch, 128, 8, 8) # changed input channels to match concatenated input with residual
        nn.BatchNorm2d(128),
        nn.ReLU(inplace=True)
    )


    self.conv5 = nn.Sequential(
        nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1), #(batch, 256, 4, 4)
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True)
    )


    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256*4*4, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU(inplace=True),
        nn.Linear(4096, num_classes)
    )

    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


  def forward(self, x):
    conv1 = self.conv1(x) #(batch, 16, 16, 16)
    conv2 = self.conv2(conv1) #(batch, 32, 16, 16)
    conv3 = self.conv3(conv2) #(batch, 64, 8, 8)
    conv2_pooled = self.pool(conv2)
    conv4_res= torch.cat([conv3, conv2_pooled], dim=1) # (batch, 128, 8, 8)
    conv4 = self.conv4(conv4_res) #(batch, 128, 8, 8)
    conv5 = self.conv5(conv4) #(batch, 256, 4, 4)

    classification = self.classifier(conv5)
    return classification


In [ ]:
# Write your code here
import torch.optim as optim

model = CNNResidual(num_classes=3) #already defined in the previous code
LEARNING_RATE = .001
NUM_EPOCHS= 10
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):
  train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
  val_loss, val_acc = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)
  train_accuracies.append(train_acc)
  val_accuracies.append(val_acc)

  print(f"Epoch[{epoch+1}/{NUM_EPOCHS}]: Training Loss: {train_loss} | Validation Loss: {val_loss} | Training Accuracy: {train_acc} | Validation Accuracy: {val_acc}")

In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, NUM_EPOCHS+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, NUM_EPOCHS+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, NUM_EPOCHS+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, NUM_EPOCHS+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()
